# 2 · OIE extraction with Gemini  (per article)

Free-form open information extraction on one article at a time:
1. **Entities + types** — Gemini invents the type label that best fits each entity, then defines
   each distinct type.
2. **Triples** — relations among those entities (subject/object must be extracted entities).

Set `ARTICLE_ID` below to pick an article from `dataset/articles/`, or run the final **batch**
cell to process them all. Outputs go to `results/<ARTICLE_ID>/oie_entities.json` and
`oie_triples.json`.

In [5]:
# --- Config -------------------------------------------------------------------------------
from pathlib import Path

ARTICLE_ID   = "art2"                    # which article to process (or use the batch cell below)
DATASET_DIR  = Path("dataset/articles")  # inputs:  <ARTICLE_ID>.txt
RESULTS_DIR  = Path("results")           # outputs: results/<ARTICLE_ID>/oie_*.json
GEMINI_MODEL = "gemini-2.5-flash"
print("article:", ARTICLE_ID, "| dataset:", DATASET_DIR, "| results:", RESULTS_DIR)

article: art2 | dataset: dataset/articles | results: results


In [2]:
# --- Gemini client -----------------------------------------------------------------------
%pip install -q google-genai

import os, json, time
from google import genai
from google.genai import types as genai_types
from IPython.display import Markdown, display

GEMINI_API_KEY = "" or os.environ.get("GEMINI_API_KEY") or os.environ.get("GOOGLE_API_KEY")
assert GEMINI_API_KEY, "Paste your key in GEMINI_API_KEY above, or set it in the env and restart the kernel."

client = genai.Client(api_key=GEMINI_API_KEY)
last_latency_s = None

def gemini_json(system_prompt, user_message, model=None, temperature=0):
    """Call Gemini with JSON-mode output; return the parsed object and record latency."""
    global last_latency_s
    t0 = time.perf_counter()
    resp = client.models.generate_content(
        model=model or GEMINI_MODEL,
        contents=user_message,
        config=genai_types.GenerateContentConfig(
            system_instruction=system_prompt,
            temperature=temperature,
            response_mime_type="application/json",
        ),
    )
    last_latency_s = time.perf_counter() - t0
    return json.loads(resp.text)

Note: you may need to restart the kernel to use updated packages.


## 1. Entity + type extraction

In [6]:
# Free-form entity + type extraction -> OIE_entities_gemini.json
OIE_ENTITY_SYSTEM_PROMPT = """You are an expert Entity Recognition and Typing system.
You are given a TEXT to analyze, and a REFERENCE_DATE (the article's publication date, ISO-8601) to anchor relative time expressions.
TASK: Extract every entity explicitly mentioned in the TEXT. For each entity, assign the single most appropriate type label, choosing the label freely based on what the entity is. You are NOT given a fixed set of types: invent whatever type label best describes each entity. The SAME surface text may be extracted more than once if, in different parts of the text, it clearly refers to different kinds of things (i.e. warrants different type labels). After extraction, produce a definition for every unique type label you used.
RULES:
1. Extract entities exactly as they appear; preserve original casing and punctuation in the "text" field. Do not invent, infer, or normalize the surface text of entities that are not explicitly mentioned. (Any resolved/normalized value goes in a SEPARATE field, never in "text" — see rule 7.)
2. Assign each entity a free-form type label of your own choosing. Pick the label that most precisely captures what kind of thing the entity is, given the evidence in the text. Do not restrict yourself to any predefined vocabulary.
3. Keep type labels consistent: if two entities are the same kind of thing, give them the exact same label (same spelling, same casing). Prefer a single clear noun or short noun phrase per label (e.g. "Person", "City", "Software Library", "Legal Case").
4. Choose the most specific label the text justifies. Use a general label only when the text does not support a more specific one; do not guess a specificity the evidence does not warrant.
5. Deduplicate on the (text, type) pair, not on text alone:
   - If the same surface text appears multiple times AND refers to the same kind of thing each time, include it ONCE.
   - If the same surface text refers to genuinely different kinds of things in different places (each supported by the text), include it ONCE PER DISTINCT TYPE, using the same surface text with the different type labels. Only do this when the evidence clearly supports more than one type for that exact mention text; do not split a single referent across types just because a label is uncertain.
6. "confidence" is your certainty in the assigned type, between 0 and 1.
7. TEMPORAL NORMALIZATION — the "normalized_value" field:
   - For every entity that denotes a date or time — whether absolute ("24 June", "2026"), relative, or deictic ("Wednesday", "yesterday", "the last week of June", "this weekend", "next year") — resolve it to an ISO-8601 calendar value and put it in "normalized_value". Resolve relative/deictic expressions against REFERENCE_DATE, inferring direction (past vs. future) from the narrative context.
   - Use the granularity the text justifies: "YYYY-MM-DD" for a specific day, or a reduced form ("YYYY-MM", "YYYY") when only a coarser unit is warranted. For a span, give the start of the span at the justified granularity.
   - NEVER put the normalized value in "text": the original surface form always stays verbatim in "text".
   - For every NON-temporal entity, and for any temporal expression you cannot confidently resolve, set "normalized_value" to null.
8. After extracting all entities, collect every UNIQUE type label you used. For each, provide: (a) "definition": a concise one-sentence definition describing the category in general terms (what this kind of thing is); and (b) "synonyms": a short list of 2-5 alternative names or near-synonyms for the type (aka labels; e.g. for "Musician": "singer", "recording artist", "performer"). Both must be general and independent of the specific entities in the text. Do NOT include example instances, and do NOT include contrastive "distinct from" notes.
9. The set of distinct type labels used in "entities" must be exactly the set of labels defined in "types": every label used on an entity has exactly one definition, and every defined label is used by at least one entity. A label may be used by multiple entities, but is defined only once. The same (text, type) pair must not appear more than once.
10. Return ONLY valid JSON matching the schema below. No explanations, no extra text.
OUTPUT SCHEMA:
{
  "entities": [
    { "text": "entity mention", "type": "type_label", "normalized_value": "2026-06-24", "confidence": 0.95 }
  ],
  "types": [
    { "type": "type_label", "definition": "A concise general definition of this type.", "synonyms": ["alternative name", "aka", "near-synonym"] }
  ]
}
Note: "normalized_value" is null for every non-temporal entity and for unresolvable time expressions."""


import re

def _reference_date(text):
    """Parse the 'Date: DD/MM/YYYY' dateline from the article header -> ISO-8601 (YYYY-MM-DD), or None."""
    m = re.search(r"^Date:\s*(\d{2})/(\d{2})/(\d{4})", text, re.MULTILINE)
    if not m:
        return None
    d, mth, y = m.groups()
    return f"{y}-{mth}-{d}"


def extract_entities(article_id):
    """Run Gemini free-form entity+type OIE on one article -> results/<id>/oie_entities.json."""
    text = (DATASET_DIR / f"{article_id}.txt").read_text()
    ref_date = _reference_date(text)
    print(f"[{article_id}] ~{len(text)//4} tokens; REFERENCE_DATE={ref_date}; calling {GEMINI_MODEL} ...")
    user_message = (f"REFERENCE_DATE: {ref_date}\n\nTEXT:\n{text}" if ref_date else f"TEXT:\n{text}")
    oie = gemini_json(OIE_ENTITY_SYSTEM_PROMPT, user_message)
    out_dir = RESULTS_DIR / article_id
    out_dir.mkdir(parents=True, exist_ok=True)
    (out_dir / "oie_entities.json").write_text(json.dumps(oie, indent=2, ensure_ascii=False))
    n_norm = sum(1 for e in oie.get("entities", []) if e.get("normalized_value"))
    print(f"  latency {last_latency_s:.2f}s | {len(oie.get('entities', []))} entities "
          f"({n_norm} time-normalized), {len(oie.get('types', []))} distinct types -> {out_dir / 'oie_entities.json'}")
    return oie

oie = extract_entities(ARTICLE_ID)

[art2] ~1865 tokens; REFERENCE_DATE=2026-07-03; calling gemini-2.5-flash ...
  latency 67.65s | 113 entities (16 time-normalized), 43 distinct types -> results/art2/oie_entities.json


## 2. Relation (triple) extraction

In [4]:
# OIE relation extraction (uses the entities above) -> OIE_triples_gemini.json -------------
OIE_RELATION_SYSTEM_PROMPT = """You are an expert Knowledge Graph Triple Extraction system.

You are given:
1. A TEXT to analyze.
2. A list of ENTITIES already extracted from the TEXT, each with its type.

TASK: Extract every triple (subject, relation, object) that the TEXT explicitly states or strongly implies. You are NOT given a fixed set of relations: name each relation freely based on what the TEXT expresses.

RULES:
1. Both the SUBJECT and the OBJECT of every triple MUST be entities from the ENTITIES list. Use the entity's EXACT "text" value. Do NOT introduce any new mention not in the list.
2. Name the relation with a free-form label of your own choosing. Pick the label that most precisely captures the relationship the TEXT expresses between subject and object. Do not restrict yourself to any predefined vocabulary.
3. Use a verb or verb phrase in lowerCamelCase, read as "subject RELATION object" (e.g. "foundedBy", "locatedIn", "worksFor", "marriedTo"). Keep the direction consistent with subject and object: the subject is the thing the relation is about, the object is what it points to.
4. Keep relation labels consistent: if two triples express the same kind of relationship, give them the exact same label (same spelling, same casing).
5. Extract only triples explicitly stated or strongly implied by the TEXT. Do not invent relationships that are not supported.
6. Deduplicate identical triples (same subject, relation, and object).
7. "confidence" is your certainty in the triple, between 0 and 1.
8. After extracting all triples, collect every UNIQUE relation label you used and write a concise one-sentence definition for each. The definition should describe the relationship in general terms (what it means for one thing to stand in this relation to another), independent of the specific entities in the text.
9. The set of distinct relation labels used in "triples" must be exactly the set of labels defined in "relations": every label used in a triple has exactly one definition, and every defined label is used by at least one triple. A label may be used by multiple triples, but is defined only once.
10. Return ONLY valid JSON matching the schema below. No explanations, no extra text.

OUTPUT SCHEMA:
{
  "triples": [
    { "subject": "entity text", "relation": "relationLabel", "object": "entity text", "confidence": 0.95 }
  ],
  "relations": [
    { "relation": "relationLabel", "definition": "A concise general definition of this relation." }
  ]
}"""


def extract_triples(article_id):
    """Extract relations among the already-found entities -> results/<id>/oie_triples.json.
    Every subject/object must be one of the extracted entities; the rest are flagged invalid."""
    out_dir = RESULTS_DIR / article_id
    text = (DATASET_DIR / f"{article_id}.txt").read_text()
    entities = json.loads((out_dir / "oie_entities.json").read_text()).get("entities", [])
    entities_for_prompt = [{"text": e["text"], "type": e["type"]} for e in entities]
    user_message = (f"ENTITIES:\n{json.dumps(entities_for_prompt, ensure_ascii=False, indent=2)}"
                    f"\n\nTEXT:\n{text}")
    print(f"[{article_id}] ~{len(user_message)//4} tokens; calling {GEMINI_MODEL} ...")
    obj = gemini_json(OIE_RELATION_SYSTEM_PROMPT, user_message)
    triples = obj.get("triples", [])
    entity_texts = {e["text"] for e in entities}
    valid   = [t for t in triples if t.get("subject") in entity_texts and t.get("object") in entity_texts]
    invalid = [t for t in triples if t not in valid]
    (out_dir / "oie_triples.json").write_text(json.dumps(
        {"triples": valid, "invalid": invalid, "relations": obj.get("relations", [])},
        indent=2, ensure_ascii=False))
    print(f"  {len(valid)} valid, {len(invalid)} invalid -> {out_dir / 'oie_triples.json'}")
    return valid, invalid

valid, invalid = extract_triples(ARTICLE_ID)
display(Markdown("\n".join(
    [f"**{len(valid)} valid triples**"] +
    [f"- **{t.get('subject','')}** — *{t.get('relation','')}* → **{t.get('object','')}**"
     f"  _(conf {t.get('confidence','?')})_" for t in valid])))

[art1] ~2589 tokens; calling gemini-2.5-flash ...


KeyboardInterrupt: 

## 3. Batch — process every article in the dataset

In [ ]:
# Runs entities + triples for every dataset/articles/*.txt. Re-runs overwrite existing outputs.
for fp in sorted(DATASET_DIR.glob("*.txt")):
    aid = fp.stem
    print("=" * 60, aid)
    extract_entities(aid)
    extract_triples(aid)
print("\ndone — all articles processed")